# Module 4.3: Dimensionality Reduction: PCA & UMAP for 50+ PLC Telemetry Tags
### 🇵🇱 Redukcja Wymiarowości: PCA i UMAP dla Ponad 50 Sygnałów PLC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m04_l03_pca_umap_plc_compression.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Compress high-dimensional multi-sensor arrays (temperatures, pressures, motor currents) into 2D and 3D visual health manifolds using PCA and non-linear UMAP.  
> 🇵🇱 **Streszczenie:** Zredukuj dziesiątki skorelowanych zmiennych z PLC (temperatury, ciśnienia, prądy) do czytelnych map 2D/3D stanu technicznego za pomocą PCA i UMAP.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **The '50-Gauge' Dilemma**: Modern packaging lines generate 50-200 SCADA/PLC tags simultaneously. Operators cannot monitor 50 individual dials.
2. **Principal Component Analysis (PCA)**: Projects orthogonal variance directions. The scree plot indicates how many latent components are required to capture $85\%-90\%$ of total plant variance.
3. **Machine Health Radar**: Mapping high-dimensional telemetry into a 2D coordinate plane allows real-time visualization of machine health drifting from green (nominal) to red (breakdown).


In [ ]:
# 1. Simulate 50 Correlated PLC Tags on an Automated Packaging Line
from sklearn.decomposition import PCA

np.random.seed(42)
n_cycles = 1000
n_plc_tags = 50

# Base latent operational states (Line Speed, Ambient Temp, Product Mass)
latent_state = np.random.normal(0, 1, (n_cycles, 3))
mixing_matrix = np.random.normal(0, 0.5, (3, n_plc_tags))

# 50 correlated sensor channels + sensor noise
plc_data = np.dot(latent_state, mixing_matrix) + np.random.normal(0, 0.2, (n_cycles, n_plc_tags))

# Simulate machine wear progression over the last 200 cycles
wear_vector = np.linspace(0, 3.5, 200)[:, None] * np.random.uniform(0.5, 1.5, n_plc_tags)
plc_data[-200:] += wear_vector

print(f"✓ Generated {n_cycles} cycles across {n_plc_tags} PLC tags.")


In [ ]:
# 2. PCA Scree Plot & 2D Machine Health Radar
pca = PCA(n_components=10)
pca_transformed = pca.fit_transform(plc_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree Plot
axes[0].plot(range(1, 11), np.cumsum(pca.explained_variance_ratio_)*100, marker='o', color='#0284c7', linewidth=2)
axes[0].axhline(85.0, color='#ef4444', linestyle='--', label='85% Variance Target')
axes[0].set_title("PCA Scree Plot: Cumulative Explained Variance", fontweight='bold')
axes[0].set_xlabel("Principal Components")
axes[0].set_ylabel("Variance Explained (%)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2D Machine Health Radar (PC1 vs PC2)
scatter = axes[1].scatter(pca_transformed[:, 0], pca_transformed[:, 1], c=np.arange(n_cycles), cmap='coolwarm', alpha=0.7)
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label("Production Cycle (Wear Progression)")
axes[1].set_title("Machine Health Radar (50 PLC Tags -> 2D Projection)", fontweight='bold')
axes[1].set_xlabel("Principal Component 1")
axes[1].set_ylabel("Principal Component 2")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Fit Principal Component Analysis (PCA) on a 50-channel chemical reactor telemetry array *(🇵🇱 Dopasuj analizę składowych głównych (PCA) do 50 kanałów pomiarowych reaktora chemicznego)*
- [ ] **Task 2:** Plot explained variance ratio scree plot to determine the minimum number of principal components *(🇵🇱 Wykreśl wykres osypiska wariancji wyjaśnionej, dobierając optymalną liczbę składowych)*
- [ ] **Task 3:** Project complex machine degradation trajectories onto a 2D non-linear UMAP manifold *(🇵🇱 Zrzutuj trajektorie zużywania się maszyny na dwuwymiarową mapę nieliniową UMAP)*


In [ ]:
# Exercise Challenge:
# Verify that the first 3 principal components capture at least 70% of total variance
top3_var = np.sum(pca.explained_variance_ratio_[:3]) * 100
print(f"Top 3 PCs Explained Variance: {top3_var:.1f}%")
assert top3_var >= 70.0, "PCA failed to capture primary operational variance."
print("✓ Dimensionality reduction verification passed.")
